# Exploratory Data Analysis (EDA) - RAF-DB Dataset

Notebook này thực hiện các phân tích dữ liệu, vẽ biểu đồ và kiểm tra chất lượng dữ liệu. Các phần tiền xử lý (như augmentation, chuẩn hóa) đã được chuyển sang file `utils/data_loader.py`.

In [ ]:
%matplotlib inline
import os
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# CẤU HÌNH
DATA_DIR = "data/DATASET"
TRAIN_DIR = os.path.join(DATA_DIR, "train")
TEST_DIR = os.path.join(DATA_DIR, "test")
TRAIN_CSV = "data/train_labels.csv"
TEST_CSV = "data/test_labels.csv"
OUTPUT_DIR = "outputs/eda"

os.makedirs(OUTPUT_DIR, exist_ok=True)

# Mapping label → tên cảm xúc
EMOTION_MAP = {1: "Surprise", 2: "Fear", 3: "Disgust", 4: "Happiness", 5: "Sadness", 6: "Anger", 7: "Neutral"}
EMOTION_MAP_VI = {1: "Ngạc nhiên", 2: "Sợ hãi", 3: "Ghê tởm", 4: "Vui vẻ", 5: "Buồn bã", 6: "Tức giận", 7: "Trung lập"}
EMOTION_COLORS = {1: "#FF6B6B", 2: "#845EC2", 3: "#4B8B3B", 4: "#FFD93D", 5: "#4A90D9", 6: "#FF4444", 7: "#95A5A6"}

## 1. Phân tích cấu trúc thư mục

In [ ]:
dir_data = {}
for split_name, split_dir in [("Train", TRAIN_DIR), ("Test", TEST_DIR)]:
    print(f"\n{split_name} Directory: {split_dir}")
    class_counts = {}
    if not os.path.exists(split_dir):
        print("Thư mục không tồn tại!")
        continue
        
    subdirs = sorted(os.listdir(split_dir))
    print(f"Số lượng thư mục con: {len(subdirs)}")
    
    total = 0
    for subdir in subdirs:
        subdir_path = os.path.join(split_dir, subdir)
        if os.path.isdir(subdir_path):
            count = len([f for f in os.listdir(subdir_path) if os.path.isfile(os.path.join(subdir_path, f))])
            emotion_name = EMOTION_MAP.get(int(subdir), "Unknown")
            print(f"{subdir}/ ({emotion_name}): {count:,} files")
            class_counts[int(subdir)] = count
            total += count
    
    print(f"Tổng: {total:,} files")
    dir_data[split_name.lower()] = class_counts

## 2. Kiểm tra và Tạo CSV Labels

In [ ]:
# Tự động tạo file CSV từ cấu trúc thư mục nếu chưa tồn tại
for split, csv_path in [("train", TRAIN_CSV), ("test", TEST_CSV)]:
    if not os.path.exists(csv_path):
        split_dir = os.path.join(DATA_DIR, split)
        if not os.path.exists(split_dir): continue
        print(f"Đang tự động tạo {csv_path} từ thư mục...")
        data = []
        for class_id in os.listdir(split_dir):
            class_path = os.path.join(split_dir, class_id)
            if not os.path.isdir(class_path): continue
            for img_name in os.listdir(class_path):
                if img_name.endswith(('.jpg', '.jpeg', '.png')):
                    data.append({"filename": img_name, "label": int(class_id)})
        pd.DataFrame(data).to_csv(csv_path, index=False)
        print(f"Đã tạo {csv_path}.")

csv_data = {}
for csv_name, csv_path in [("Train", TRAIN_CSV), ("Test", TEST_CSV)]:
    print(f"\n{csv_name} CSV: {csv_path}")
    if not os.path.exists(csv_path): continue
    
    df = pd.read_csv(csv_path)
    print(f"Shape: {df.shape}")
    print(df.head())
    csv_data[csv_name.lower()] = df

## 3. Trực quan hóa Phân bố nhãn

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 7))
fig.suptitle("RAF-DB: Phân bố Nhãn Cảm xúc", fontsize=18, fontweight='bold', y=1.02)

for idx, (split, ax) in enumerate(zip(["train", "test"], axes)):
    if split not in csv_data: continue
    
    df = csv_data[split]
    label_col = df.columns[-1]
    counts = df[label_col].value_counts().sort_index()
    
    labels = [f"{EMOTION_MAP[i]}\n({EMOTION_MAP_VI[i]})" for i in counts.index]
    colors = [EMOTION_COLORS[i] for i in counts.index]
    
    bars = ax.bar(labels, counts.values, color=colors, edgecolor='white', linewidth=1.5, alpha=0.9)
    for bar, val in zip(bars, counts.values):
        pct = val / len(df) * 100
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 30,
                f'{val:,}\n({pct:.1f}%)', ha='center', va='bottom', fontsize=9, fontweight='bold')
    
    ax.set_title(f'{split.upper()} Set ({len(df):,} ảnh)', fontsize=14, fontweight='bold')
    ax.set_ylabel('Số lượng ảnh')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.set_ylim(0, max(counts.values) * 1.25)
    ax.grid(axis='y', alpha=0.3, linestyle='--')

plt.tight_layout()
plt.show()

## 4. Mẫu Dữ liệu theo từng Class

In [ ]:
samples_per_class = 6
n_classes = 7
fig, axes = plt.subplots(n_classes, samples_per_class, figsize=(samples_per_class * 2.5, n_classes * 2.8))
fig.suptitle('RAF-DB: Mẫu dữ liệu theo từng cảm xúc', fontsize=18, fontweight='bold', y=1.01)

for class_idx in range(1, n_classes + 1):
    class_dir = os.path.join(TRAIN_DIR, str(class_idx))
    if not os.path.exists(class_dir): continue
    
    files = sorted(os.listdir(class_dir))
    np.random.seed(42)
    selected = np.random.choice(files, size=min(samples_per_class, len(files)), replace=False)
    
    for col_idx, fname in enumerate(selected):
        ax = axes[class_idx - 1][col_idx]
        img = cv2.imread(os.path.join(class_dir, fname))
        if img is not None:
            ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
        ax.axis('off')
        
        if col_idx == 0:
            emotion_en = EMOTION_MAP[class_idx]
            emotion_vi = EMOTION_MAP_VI[class_idx]
            ax.set_ylabel(f'{emotion_en}\n({emotion_vi})', fontsize=11, fontweight='bold', rotation=0, labelpad=80, va='center')

plt.tight_layout()
plt.show()

## 5. Khuôn mặt Trung bình (Mean Face)

In [ ]:
fig, axes = plt.subplots(1, 7, figsize=(21, 4))
fig.suptitle('Khuôn mặt Trung bình theo từng Cảm xúc', fontsize=16, fontweight='bold')

for class_idx in range(1, 8):
    class_dir = os.path.join(TRAIN_DIR, str(class_idx))
    if not os.path.exists(class_dir): continue
    
    files = os.listdir(class_dir)
    mean_face = np.zeros((100, 100, 3), dtype=np.float64)
    count = 0
    for fname in files:
        img = cv2.imread(os.path.join(class_dir, fname))
        if img is not None:
            mean_face += cv2.cvtColor(img, cv2.COLOR_BGR2RGB).astype(np.float64)
            count += 1
    
    if count > 0:
        mean_face = (mean_face / count).astype(np.uint8)
    
    ax = axes[class_idx - 1]
    ax.imshow(mean_face)
    ax.set_title(f'{EMOTION_MAP[class_idx]}\n({count} ảnh)', fontsize=10, fontweight='bold')
    ax.axis('off')

plt.tight_layout()
plt.show()

## 6. Thống kê Pixel và Histogram

In [ ]:
pixel_sum = np.zeros(3, dtype=np.float64)
pixel_sq_sum = np.zeros(3, dtype=np.float64)
total_pixels = 0
all_intensities = []
sample_count = 0

for split_dir in [TRAIN_DIR, TEST_DIR]:
    if not os.path.exists(split_dir): continue
    for class_dir in sorted(os.listdir(split_dir)):
        class_path = os.path.join(split_dir, class_dir)
        if not os.path.isdir(class_path): continue
        for fname in os.listdir(class_path):
            fpath = os.path.join(class_path, fname)
            img = cv2.imread(fpath)
            if img is None: continue
            
            img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            img_float = img_rgb.astype(np.float64)
            total_pixels += img_float.shape[0] * img_float.shape[1]
            pixel_sum += img_float.reshape(-1, 3).sum(axis=0)
            pixel_sq_sum += (img_float.reshape(-1, 3) ** 2).sum(axis=0)
            
            if sample_count < 2000:
                gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
                all_intensities.extend(gray.flatten().tolist()[::10])
                sample_count += 1

if total_pixels > 0:
    mean = pixel_sum / total_pixels
    std = np.sqrt(pixel_sq_sum / total_pixels - mean ** 2)

    print(f"Pixel Statistics (RGB):")
    print(f"Mean: R={mean[0]:.2f}, G={mean[1]:.2f}, B={mean[2]:.2f}")
    print(f"Std:  R={std[0]:.2f}, G={std[1]:.2f}, B={std[2]:.2f}")

    fig, ax = plt.subplots(figsize=(12, 6))
    ax.hist(all_intensities, bins=64, color='#4A90D9', edgecolor='white', alpha=0.8, density=True)
    ax.set_title('Phân bố Cường độ Pixel (Grayscale)')
    mean_val = np.mean(all_intensities)
    ax.axvline(mean_val, color='#FF4444', linestyle='--', linewidth=2, label=f'Mean = {mean_val:.1f}')
    ax.legend()
    plt.show()

## 7. Phân tích Mất cân bằng dữ liệu (Imbalance)

In [ ]:
if "train" in csv_data:
    df = csv_data["train"]
    label_col = df.columns[-1]
    counts = df[label_col].value_counts().sort_index()
    
    max_count, min_count = counts.max(), counts.min()
    print(f"Class lớn nhất: {EMOTION_MAP[counts.idxmax()]} = {max_count:,}")
    print(f"Class nhỏ nhất: {EMOTION_MAP[counts.idxmin()]} = {min_count:,}")
    print(f"Imbalance Ratio (max/min): {max_count/min_count:.1f}:1")
    
    fig, axes = plt.subplots(1, 2, figsize=(18, 7))
    labels = [f"{EMOTION_MAP[i]}\n{EMOTION_MAP_VI[i]}" for i in counts.index]
    colors = [EMOTION_COLORS[i] for i in counts.index]
    
    axes[0].pie(counts.values, labels=labels, autopct='%1.1f%%', colors=colors, startangle=90)
    axes[0].set_title('Tỉ lệ phân bố các cảm xúc (Train)')
    
    ratios = [(max_count / c) for c in counts.values]
    bars = axes[1].barh([EMOTION_MAP[i] for i in counts.index], ratios, color=colors, edgecolor='white', alpha=0.9)
    axes[1].set_title('Mức độ Mất cân bằng (Imbalance Ratio)')
    axes[1].axvline(x=1, color='green', linestyle='--', linewidth=2, label='Balanced (1:1)')
    axes[1].legend()
    
    plt.tight_layout()
    plt.show()